# Gradient Boosting — Tiempo de Carrera de 5K

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** Machine Learning · Supervisado · Regresión · Gradient Boosting

---

### Objetivo

Estimar el **tiempo que tarda un corredor en completar una carrera popular de 5 km** a partir de su volumen de entrenamiento semanal, su frecuencia cardíaca en reposo y su tipo de entrenamiento, usando un **Gradient Boosting Regressor**.

### Contexto de negocio

**El cliente:** una app de running que quiere dar a cada corredor una estimación de tiempo de carrera antes del día de la prueba, a partir de datos que ya registra (kilómetros semanales, frecuencia cardíaca en reposo).

**El problema:** los corredores sobrestiman o infravaloran su tiempo objetivo constantemente, lo que genera salidas mal planificadas (demasiado rápido al principio, "muro" al final).

**La pregunta:** ¿bastan tres variables fáciles de registrar (volumen, forma física básica, tipo de entrenamiento) para predecir el tiempo de carrera con un margen de error útil?

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

# Estilo visual — sistema de color validado (consejo UX/UI Data)
BACKGROUND    = '#fbfbfb'
PURPLE        = '#7a7bff'   # único color de énfasis (1 por gráfico) / serie única en scatter y líneas
PURPLE_LIGHT  = '#9b9cff'   # EDA de una sola serie (histogramas) -- PURPLE fuerte queda solo para el enfasis
POSITIVE      = '#6b8158'   # exclusivo signo positivo
NEGATIVE      = '#c34031'   # exclusivo signo negativo
NEUTRAL_BAR   = '#d9d9d9'   # barras/áreas de contexto (siempre con etiqueta de valor)
NEUTRAL_LINE  = '#8f8c9e'   # líneas de contexto (más contraste que NEUTRAL_BAR)
CONTEXT_LINES = [NEUTRAL_LINE, '#a89a8a', '#7d94a8']   # gama fija para 2+ líneas de contexto en un mismo gráfico
INK           = '#111111'
MUTED         = '#707070'

DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "borja_diverging", ["#c34031", "#e0a89f", "#f0ede8", "#b7c2a9", "#6b8158"]
)
SEQUENTIAL_GREEN = LinearSegmentedColormap.from_list(
    "borja_sequential", [BACKGROUND, POSITIVE]
)

def color_annotations(ax, values, threshold, dark="#ffffff", light=INK):
    """Recolorea el texto de un heatmap celda a celda según su magnitud."""
    for text, value in zip(ax.texts, np.asarray(values).flatten()):
        text.set_color(dark if abs(value) >= threshold else light)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

## 2. Carga de datos

In [2]:
datos_corredores = [
    {"Km_Semanales": 31.84, "FC_Reposo": 68.61, "Tipo_Entrenamiento": 1, "Tiempo_5k": 24.32},
    {"Km_Semanales": 57.78, "FC_Reposo": 58.07, "Tipo_Entrenamiento": 1, "Tiempo_5k": 20.35},
    {"Km_Semanales": 47.94, "FC_Reposo": 63.30, "Tipo_Entrenamiento": 2, "Tiempo_5k": 26.54},
    {"Km_Semanales": 41.94, "FC_Reposo": 58.33, "Tipo_Entrenamiento": 1, "Tiempo_5k": 25.43},
    {"Km_Semanales": 22.01, "FC_Reposo": 74.00, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.95},
    {"Km_Semanales": 22.11, "FC_Reposo": 72.58, "Tipo_Entrenamiento": 2, "Tiempo_5k": 33.56},
    {"Km_Semanales": 17.55, "FC_Reposo": 71.30, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.91},
    {"Km_Semanales": 53.95, "FC_Reposo": 58.55, "Tipo_Entrenamiento": 1, "Tiempo_5k": 20.61},
    {"Km_Semanales": 42.00, "FC_Reposo": 62.13, "Tipo_Entrenamiento": 1, "Tiempo_5k": 23.36},
    {"Km_Semanales": 46.86, "FC_Reposo": 60.01, "Tipo_Entrenamiento": 2, "Tiempo_5k": 26.24},
    {"Km_Semanales": 16.00, "FC_Reposo": 72.33, "Tipo_Entrenamiento": 1, "Tiempo_5k": 32.14},
    {"Km_Semanales": 58.65, "FC_Reposo": 53.64, "Tipo_Entrenamiento": 2, "Tiempo_5k": 24.28},
    {"Km_Semanales": 52.01, "FC_Reposo": 58.74, "Tipo_Entrenamiento": 2, "Tiempo_5k": 24.58},
    {"Km_Semanales": 24.51, "FC_Reposo": 72.63, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.68},
    {"Km_Semanales": 23.18, "FC_Reposo": 68.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.11},
    {"Km_Semanales": 23.25, "FC_Reposo": 68.10, "Tipo_Entrenamiento": 1, "Tiempo_5k": 29.89},
    {"Km_Semanales": 28.71, "FC_Reposo": 69.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 28.16},
    {"Km_Semanales": 28.74, "FC_Reposo": 66.21, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.52},
    {"Km_Semanales": 34.42, "FC_Reposo": 67.33, "Tipo_Entrenamiento": 2, "Tiempo_5k": 29.56},
    {"Km_Semanales": 28.09, "FC_Reposo": 65.61, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.42},
    {"Km_Semanales": 42.45, "FC_Reposo": 61.26, "Tipo_Entrenamiento": 2, "Tiempo_5k": 27.60},
    {"Km_Semanales": 21.23, "FC_Reposo": 71.60, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.56},
    {"Km_Semanales": 28.19, "FC_Reposo": 66.86, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.60},
    {"Km_Semanales": 31.49, "FC_Reposo": 65.34, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.41},
    {"Km_Semanales": 35.53, "FC_Reposo": 63.85, "Tipo_Entrenamiento": 2, "Tiempo_5k": 29.55},
    {"Km_Semanales": 50.81, "FC_Reposo": 58.74, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.31},
    {"Km_Semanales": 38.35, "FC_Reposo": 66.24, "Tipo_Entrenamiento": 1, "Tiempo_5k": 25.96},
    {"Km_Semanales": 38.07, "FC_Reposo": 62.47, "Tipo_Entrenamiento": 1, "Tiempo_5k": 26.54},
    {"Km_Semanales": 41.76, "FC_Reposo": 60.10, "Tipo_Entrenamiento": 1, "Tiempo_5k": 25.29},
    {"Km_Semanales": 17.02, "FC_Reposo": 71.07, "Tipo_Entrenamiento": 2, "Tiempo_5k": 34.69},
    {"Km_Semanales": 42.44, "FC_Reposo": 59.86, "Tipo_Entrenamiento": 1, "Tiempo_5k": 24.36},
    {"Km_Semanales": 22.66, "FC_Reposo": 70.08, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.82},
    {"Km_Semanales": 17.87, "FC_Reposo": 72.33, "Tipo_Entrenamiento": 2, "Tiempo_5k": 34.90},
    {"Km_Semanales": 58.05, "FC_Reposo": 55.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 24.16},
    {"Km_Semanales": 56.17, "FC_Reposo": 57.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 24.41},
    {"Km_Semanales": 29.97, "FC_Reposo": 68.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 28.53},
    {"Km_Semanales": 28.70, "FC_Reposo": 69.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 33.22},
    {"Km_Semanales": 19.19, "FC_Reposo": 73.08, "Tipo_Entrenamiento": 2, "Tiempo_5k": 35.12},
    {"Km_Semanales": 27.75, "FC_Reposo": 68.35, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.90},
    {"Km_Semanales": 35.15, "FC_Reposo": 66.86, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.35},
    {"Km_Semanales": 20.21, "FC_Reposo": 71.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.22},
    {"Km_Semanales": 34.78, "FC_Reposo": 67.33, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.02},
    {"Km_Semanales": 16.14, "FC_Reposo": 74.00, "Tipo_Entrenamiento": 1, "Tiempo_5k": 32.22},
    {"Km_Semanales": 55.40, "FC_Reposo": 56.19, "Tipo_Entrenamiento": 2, "Tiempo_5k": 25.10},
    {"Km_Semanales": 27.42, "FC_Reposo": 69.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 32.11},
    {"Km_Semanales": 44.52, "FC_Reposo": 59.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 27.32},
    {"Km_Semanales": 29.35, "FC_Reposo": 67.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 29.35},
    {"Km_Semanales": 42.12, "FC_Reposo": 62.35, "Tipo_Entrenamiento": 1, "Tiempo_5k": 24.31},
    {"Km_Semanales": 40.51, "FC_Reposo": 61.19, "Tipo_Entrenamiento": 1, "Tiempo_5k": 25.53},
    {"Km_Semanales": 23.32, "FC_Reposo": 70.35, "Tipo_Entrenamiento": 2, "Tiempo_5k": 33.32},
    {"Km_Semanales": 56.12, "FC_Reposo": 55.31, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.11},
    {"Km_Semanales": 27.63, "FC_Reposo": 68.31, "Tipo_Entrenamiento": 2, "Tiempo_5k": 33.35},
    {"Km_Semanales": 53.11, "FC_Reposo": 56.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 25.63},
    {"Km_Semanales": 25.32, "FC_Reposo": 70.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.12},
    {"Km_Semanales": 51.12, "FC_Reposo": 58.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.35},
    {"Km_Semanales": 25.13, "FC_Reposo": 70.19, "Tipo_Entrenamiento": 2, "Tiempo_5k": 34.31},
    {"Km_Semanales": 54.31, "FC_Reposo": 56.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 25.53},
    {"Km_Semanales": 38.32, "FC_Reposo": 63.31, "Tipo_Entrenamiento": 2, "Tiempo_5k": 29.31},
    {"Km_Semanales": 41.62, "FC_Reposo": 60.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 25.12},
    {"Km_Semanales": 45.31, "FC_Reposo": 59.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 23.85},
    {"Km_Semanales": 23.11, "FC_Reposo": 70.35, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.51},
    {"Km_Semanales": 42.12, "FC_Reposo": 61.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 28.32},
    {"Km_Semanales": 20.31, "FC_Reposo": 71.35, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.62},
    {"Km_Semanales": 52.35, "FC_Reposo": 57.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.11},
    {"Km_Semanales": 34.12, "FC_Reposo": 65.31, "Tipo_Entrenamiento": 1, "Tiempo_5k": 27.52},
    {"Km_Semanales": 16.63, "FC_Reposo": 73.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 36.31},
    {"Km_Semanales": 51.11, "FC_Reposo": 58.35, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.45},
    {"Km_Semanales": 23.35, "FC_Reposo": 70.63, "Tipo_Entrenamiento": 2, "Tiempo_5k": 33.63},
    {"Km_Semanales": 19.43, "FC_Reposo": 72.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.11},
    {"Km_Semanales": 53.63, "FC_Reposo": 56.45, "Tipo_Entrenamiento": 2, "Tiempo_5k": 25.13},
    {"Km_Semanales": 18.11, "FC_Reposo": 72.19, "Tipo_Entrenamiento": 2, "Tiempo_5k": 35.31},
    {"Km_Semanales": 52.12, "FC_Reposo": 57.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.62},
    {"Km_Semanales": 42.32, "FC_Reposo": 60.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 28.19},
    {"Km_Semanales": 19.43, "FC_Reposo": 71.62, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.11},
    {"Km_Semanales": 48.35, "FC_Reposo": 59.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 23.11},
    {"Km_Semanales": 16.32, "FC_Reposo": 73.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 35.53},
    {"Km_Semanales": 55.12, "FC_Reposo": 56.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.43},
    {"Km_Semanales": 34.31, "FC_Reposo": 65.32, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.19},
    {"Km_Semanales": 20.12, "FC_Reposo": 71.62, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.62},
    {"Km_Semanales": 51.11, "FC_Reposo": 58.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.51},
    {"Km_Semanales": 35.32, "FC_Reposo": 65.31, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.53},
    {"Km_Semanales": 18.12, "FC_Reposo": 72.35, "Tipo_Entrenamiento": 1, "Tiempo_5k": 32.11},
    {"Km_Semanales": 54.31, "FC_Reposo": 56.63, "Tipo_Entrenamiento": 2, "Tiempo_5k": 25.31},
    {"Km_Semanales": 24.11, "FC_Reposo": 70.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.35},
    {"Km_Semanales": 51.12, "FC_Reposo": 57.51, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.43},
    {"Km_Semanales": 33.32, "FC_Reposo": 65.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.35},
    {"Km_Semanales": 19.11, "FC_Reposo": 72.19, "Tipo_Entrenamiento": 2, "Tiempo_5k": 35.51},
    {"Km_Semanales": 54.62, "FC_Reposo": 56.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.32},
    {"Km_Semanales": 35.53, "FC_Reposo": 65.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.51},
    {"Km_Semanales": 21.11, "FC_Reposo": 71.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.12},
    {"Km_Semanales": 48.12, "FC_Reposo": 59.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 26.51},
    {"Km_Semanales": 24.31, "FC_Reposo": 70.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.45},
    {"Km_Semanales": 51.12, "FC_Reposo": 57.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.35},
    {"Km_Semanales": 35.32, "FC_Reposo": 65.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.51},
    {"Km_Semanales": 19.35, "FC_Reposo": 72.19, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.43},
    {"Km_Semanales": 53.62, "FC_Reposo": 56.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.62},
    {"Km_Semanales": 34.31, "FC_Reposo": 65.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 31.12},
    {"Km_Semanales": 20.35, "FC_Reposo": 71.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.31},
    {"Km_Semanales": 49.12, "FC_Reposo": 59.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 26.35},
    {"Km_Semanales": 23.31, "FC_Reposo": 70.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.45},
    {"Km_Semanales": 52.12, "FC_Reposo": 57.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.11},
    {"Km_Semanales": 34.32, "FC_Reposo": 65.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.85},
    {"Km_Semanales": 18.35, "FC_Reposo": 72.19, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.62},
    {"Km_Semanales": 54.62, "FC_Reposo": 56.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.11},
    {"Km_Semanales": 35.31, "FC_Reposo": 65.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.45},
    {"Km_Semanales": 21.35, "FC_Reposo": 71.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.11},
    {"Km_Semanales": 48.12, "FC_Reposo": 59.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 26.62},
    {"Km_Semanales": 24.31, "FC_Reposo": 70.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.35},
    {"Km_Semanales": 51.12, "FC_Reposo": 57.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.11},
    {"Km_Semanales": 34.32, "FC_Reposo": 65.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.62},
    {"Km_Semanales": 19.35, "FC_Reposo": 72.19, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.35},
    {"Km_Semanales": 53.62, "FC_Reposo": 56.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.43},
    {"Km_Semanales": 35.31, "FC_Reposo": 65.12, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.31},
    {"Km_Semanales": 20.35, "FC_Reposo": 71.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.12},
    {"Km_Semanales": 49.12, "FC_Reposo": 59.43, "Tipo_Entrenamiento": 2, "Tiempo_5k": 26.11},
    {"Km_Semanales": 23.31, "FC_Reposo": 70.11, "Tipo_Entrenamiento": 1, "Tiempo_5k": 30.51},
    {"Km_Semanales": 52.12, "FC_Reposo": 57.32, "Tipo_Entrenamiento": 1, "Tiempo_5k": 22.35},
    {"Km_Semanales": 34.32, "FC_Reposo": 65.11, "Tipo_Entrenamiento": 2, "Tiempo_5k": 30.85},
    {"Km_Semanales": 18.35, "FC_Reposo": 72.19, "Tipo_Entrenamiento": 1, "Tiempo_5k": 31.45},
    {"Km_Semanales": 54.62, "FC_Reposo": 56.12, "Tipo_Entrenamiento": 1, "Tiempo_5k": 21.11}
]

In [3]:
df_running = pd.DataFrame(datos_corredores)

print(f"Filas: {df_running.shape[0]}")
print(f"Columnas: {df_running.shape[1]}")

df_running.head()

Filas: 120
Columnas: 4


,Km_Semanales,FC_Reposo,Tipo_Entrenamiento,Tiempo_5k
0,31.84,68.61,1,24.32
1,57.78,58.07,1,20.35
2,47.94,63.30,2,26.54
3,41.94,58.33,1,25.43
4,22.01,74.00,1,31.95


**`Km_Semanales`** (volumen de entrenamiento), **`FC_Reposo`** (frecuencia cardíaca en reposo, un proxy de forma física cardiovascular) y **`Tipo_Entrenamiento`** (1 = con series/calidad, 2 = solo rodaje continuo) son las tres variables de entrada. **`Tiempo_5k`** (minutos) es el objetivo a predecir.

### Calidad del dato: filas casi duplicadas

Antes de modelar, comprobamos si hay corredores repetidos o casi idénticos — algo habitual en datasets generados sintéticamente, y que puede inflar o distorsionar la evaluación si una fila (o su casi-gemela) acaba a la vez en train y en test.

In [4]:
duplicados_exactos = df_running.duplicated().sum()
duplicados_por_variables = df_running.duplicated(
    subset=["Km_Semanales", "FC_Reposo", "Tipo_Entrenamiento"]
).sum()

print(f"Filas exactamente duplicadas: {duplicados_exactos}")
print(f"Filas con las mismas variables de entrada (distinto Tiempo_5k): {duplicados_por_variables}")

Filas exactamente duplicadas: 2
Filas con las mismas variables de entrada (distinto Tiempo_5k): 16


Hay **2 filas idénticas** y otras **16 filas** que comparten exactamente los mismos valores de `Km_Semanales`, `FC_Reposo` y `Tipo_Entrenamiento` pero con un `Tiempo_5k` ligeramente distinto — se concentran en el tramo final del dataset, en bloques que se repiten con pequeñas variaciones. Son compatibles con datos generados sintéticamente añadiendo ruido a una misma plantilla, más que con 120 corredores realmente distintos. Lo tenemos en cuenta más adelante al evaluar el modelo.

## 3. Train / Test split

In [5]:
X = df_running[["Km_Semanales", "FC_Reposo", "Tipo_Entrenamiento"]]
y = df_running["Tiempo_5k"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Usamos un `train_test_split` aleatorio simple: `Tiempo_5k` es una variable **continua**, así que no aplica el muestreo estratificado (que exige una variable categórica para mantener sus proporciones en cada partición) salvo que se estratifique por `Tipo_Entrenamiento` — algo razonable si esa variable estuviera muy desbalanceada, pero aquí los dos tipos de entrenamiento están representados de forma comparable en el dataset.

## 4. Modelo — Gradient Boosting Regressor

In [6]:
modelo_running = GradientBoostingRegressor(
    n_estimators=60,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
)

modelo_running.fit(X_train, y_train)
y_pred = modelo_running.predict(X_test)

## 5. Evaluación

In [7]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print("--- RESULTADOS DE LA MODELIZACIÓN ---")
print(f"MAE  (Error Absoluto Medio): {mae:.2f} minutos")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.2f} minutos")
print(f"R²   (varianza explicada): {r2:.4f} ({r2*100:.1f}%)")
print(f"MAPE (error porcentual medio): {mape:.2f}%")

--- RESULTADOS DE LA MODELIZACIÓN ---
MAE  (Error Absoluto Medio): 0.90 minutos
RMSE (Raíz del Error Cuadrático Medio): 1.43 minutos
R²   (varianza explicada): 0.7966 (79.7%)
MAPE (error porcentual medio): 3.11%


- **MAE = 0,90 minutos:** en promedio, la predicción se desvía menos de un minuto del tiempo real — si el modelo predice 25:00, el corredor suele terminar entre ~24:06 y ~25:54.
- **RMSE = 1,43 minutos**, claramente por encima del MAE: hay algunos corredores donde el modelo falla bastante más que en el caso típico (el error al cuadrado penaliza esos casos).
- **R² = 0,7966:** las tres variables explican cerca del **80%** de la variabilidad del tiempo de carrera — el 20% restante depende de factores no incluidos (edad, terreno, condiciones del día, genética).
- **MAPE = 3,11%:** de media, el modelo se equivoca en poco más del 3% del tiempo real de cada corredor — un margen bajo, independiente de la escala.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(x=y_test, y=y_pred, color=PURPLE, s=80, edgecolor="white", alpha=0.65)
plt.plot(
    [y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
    color=INK, linestyle="--", lw=2, label="Predicción perfecta",
)
plt.title("Validación: tiempo real vs. predicho (test set)", color=INK)
plt.xlabel("Tiempo real (min)")
plt.ylabel("Tiempo predicho (min)")
plt.legend(frameon=False)
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 6. ¿El resultado depende de las filas casi duplicadas?

Repetimos el mismo entrenamiento y evaluación, esta vez eliminando las filas cuyas variables de entrada están duplicadas, para comprobar si el R² de 0,7966 depende de que alguna fila "gemela" cayera a la vez en train y en test.

In [9]:
df_sin_duplicados = df_running.drop_duplicates(
    subset=["Km_Semanales", "FC_Reposo", "Tipo_Entrenamiento"], keep="first"
)

X2 = df_sin_duplicados[["Km_Semanales", "FC_Reposo", "Tipo_Entrenamiento"]]
y2 = df_sin_duplicados["Tiempo_5k"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

modelo_sin_dup = GradientBoostingRegressor(
    n_estimators=60, learning_rate=0.1, max_depth=3, random_state=42
)
modelo_sin_dup.fit(X2_train, y2_train)
y2_pred = modelo_sin_dup.predict(X2_test)

r2_sin_dup = r2_score(y2_test, y2_pred)
mae_sin_dup = mean_absolute_error(y2_test, y2_pred)

print(f"Filas tras eliminar duplicados por variables de entrada: {len(df_sin_duplicados)} (de {len(df_running)})")
print(f"R²  con duplicados:    {r2:.4f}")
print(f"R²  sin duplicados:    {r2_sin_dup:.4f}")
print(f"MAE con duplicados:    {mae:.2f} min")
print(f"MAE sin duplicados:    {mae_sin_dup:.2f} min")

Filas tras eliminar duplicados por variables de entrada: 104 (de 120)
R²  con duplicados:    0.7966
R²  sin duplicados:    0.8774
MAE con duplicados:    0.90 min
MAE sin duplicados:    0.74 min


El resultado **mejora** al quitar los duplicados (R² 0,877 frente a 0,797; MAE 0,74 frente a 0,90 minutos), no empeora — es decir, esas filas casi-gemelas no estaban "regalando" acierto por fuga de información entre train y test, sino actuando como **ruido de etiqueta**: mismos valores de entrada con un `Tiempo_5k` ligeramente distinto confunden al modelo sobre qué relación aprender. El resultado original (R²=0,7966) es honesto y no depende de una fuga de datos, pero probablemente sería aún mejor con datos de corredores genuinamente distintos en vez de variaciones sintéticas de una misma plantilla.

## 7. Conclusión

> **El hallazgo:** con solo 3 variables fáciles de registrar (volumen de entrenamiento, frecuencia cardíaca en reposo, tipo de entrenamiento), el modelo predice el tiempo de una carrera de 5K con un error medio inferior a 1 minuto (MAPE 3,1%) — explicando el 80% de la variabilidad entre corredores.

**Recomendaciones:**

1. **Aplicable ya en una app de running:** con datos que la mayoría de apps ya registran (ritmo semanal, pulsaciones en reposo), se puede dar una estimación de tiempo de carrera útil para planificar el ritmo de salida.
2. **Para mejorar el modelo:** el 20% de varianza no explicada probablemente vendría de variables no incluidas — edad, terreno de la carrera, temperatura, o historial de lesiones. Añadirlas sería el siguiente paso antes de escalar el modelo a producción.
3. **Límite honesto:** una parte del dataset está compuesta por filas casi duplicadas (mismo perfil de entrenamiento, tiempo ligeramente distinto), lo que sugiere una generación sintética por plantillas repetidas más que 120 corredores reales e independientes. El modelo funciona igual de bien (de hecho mejor) sin ellas, pero conviene tenerlo presente antes de generalizar estas métricas a datos de corredores reales.